In [24]:
from pathlib import Path

import kagglehub

# Kaggle download directories vary based on version, so local paths aren't always consistent
COMPETITION_SLUG = "nvidia-nemotron-model-reasoning-challenge"
LEGACY_KAGGLE_DIR = Path("/kaggle/input/competitions") / COMPETITION_SLUG
STANDARD_KAGGLE_DIR = Path("/kaggle/input") / COMPETITION_SLUG

def _train_csv_in(d: Path) -> bool:
    return (d / "train.csv").is_file()


if _train_csv_in(LEGACY_KAGGLE_DIR):
    DATA_DIR = LEGACY_KAGGLE_DIR
elif _train_csv_in(STANDARD_KAGGLE_DIR):
    DATA_DIR = STANDARD_KAGGLE_DIR
else:
    DATA_DIR = Path(kagglehub.competition_download(COMPETITION_SLUG))


In [ ]:
import polars as pl
import 
train = pl.read_csv(DATA_DIR / "train.csv")

train.head()

id,prompt,answer
str,str,str
"""00066667""","""In Alice's Wonderland, a secre…","""10010111"""
"""000b53cf""","""In Alice's Wonderland, a secre…","""01000011"""
"""00189f6a""","""In Alice's Wonderland, secret …","""cat imagines book"""
"""001b24c4""","""In Alice's Wonderland, numbers…","""XXXVIII"""
"""001c63cb""","""In Alice's Wonderland, secret …","""wizard creates secret"""


In [26]:
print('Training Data:')
print(f'n={train.shape[0]}')

Training Data:
n=9500


Code to view a few different puzzles

In [27]:
EXAMPLE_NUMS = range(10, 1500, 119)
for EXAMPLE_NUM in EXAMPLE_NUMS:
    print(f'Prompt:\n{train['prompt'][EXAMPLE_NUM]}')
    print()
    print(f'Answer:\n{train['answer'][EXAMPLE_NUM]}')
    print()


Prompt:
In Alice's Wonderland, a secret unit conversion is applied to measurements. For example:
32.58 m becomes 26.62
10.9 m becomes 8.90
17.86 m becomes 14.59
Now, convert the following measurement: 13.0 m

Answer:
10.62

Prompt:
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
10010110 -> 01011010
10010001 -> 01000110
10100001 -> 10000110
11010000 -> 01000011
10001000 -> 00100010
11110001 -> 11000111
01100001 -> 10000101
00000001 -> 00000100
00101010 -> 10101000

Now, determine the output for: 11000101

Answer:
00010111

Prompt:
In Alice's Wonderland, the gravitational constant has been secretly changed. Here are some example observations:
For t = 3.13s, distance = 92.92 m
For t = 3.98s, distance = 150.24 m
For t = 1.39s, distance = 18.33 m
For t = 1.16s, distance = 12.76 m
N

# Insights #

There seem to be only a couple different prompts prompts that a reworded over and over. All the puzzles seem to test model reasoning with transformations specifically.

One interesting thing is that the puzzles with the numeral system seem to all use the Roman numeral system. I'm unsure whether the testing data will be the same. I wrote some code below to look at more of these numeral puzzles and it seems like all the training data follows the Roman system.

In [28]:
numeral_examples = train.filter(pl.col('prompt').str.contains('(?i)numeral system'))
numeral_examples = numeral_examples[-10:]

for row in numeral_examples.iter_rows(named=True):
    print(f'Prompt:\n{row['prompt']}')
    print()
    print(f'Answer:\n{row['answer']}')
    print()

Prompt:
In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:
2 -> II
32 -> XXXII
32 -> XXXII
87 -> LXXXVII
Now, write the number 98 in the Wonderland numeral system.

Answer:
XCVIII

Prompt:
In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:
91 -> XCI
77 -> LXXVII
64 -> LXIV
36 -> XXXVI
51 -> LI
Now, write the number 3 in the Wonderland numeral system.

Answer:
III

Prompt:
In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:
3 -> III
59 -> LIX
40 -> XL
Now, write the number 17 in the Wonderland numeral system.

Answer:
XVII

Prompt:
In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:
59 -> LIX
76 -> LXXVI
39 -> XXXIX
86 -> LXXXVI
Now, write the number 1 in the Wonderland numeral system.

Answer:
I

Prompt:
In Alice's Wonderlan

In [ ]:
TRANSFORMATION_RULE = r'(?i)transformation rule'

CATEGORIES: list[tuple[str, str] | tuple[str, str, str]] = [
    ('numerals', r'(?i)numeral system'),
    ('unit conversion', r'(?i)unit conversion'),
    ('bit manipulation', r'(?i)bit manipulation'),
    ('gravity', r'(?i)gravitational constant'),
    ('encryption', r'(?i)encryption'),
    ('equation_symbols', TRANSFORMATION_RULE, 'no_digits'),
    ('equation_numeric', TRANSFORMATION_RULE, 'has_digits'),
]

subsets = {}
for spec in CATEGORIES:
    if len(spec) == 2:
        name, pattern = spec
        subsets[name] = train.filter(pl.col('prompt').str.contains(pattern))
    else:
        name, pattern, mode = spec
        base = train.filter(pl.col('prompt').str.contains(pattern))
        if mode == 'no_digits':
            subsets[name] = base.filter(~pl.col('prompt').str.contains(r'\d'))
        elif mode == 'has_digits':
            subsets[name] = base.filter(pl.col('prompt').str.contains(r'\d'))
        else:
            raise ValueError(f'unknown category mode: {mode!r}')

sets = list(subsets.values())
lengths = [s.shape[0] for s in sets]
print(lengths)
print(f"""Total number of examples: {train.shape[0]}
Sum of subsets: {sum(lengths)}""")

[1576, 1594, 1602, 1597, 1576, 823, 732]
Total number of examples: 9500
Sum of subsets: 9500


In [30]:
for cat, puzzles in subsets.items():
    print(f"{cat}: {puzzles.height} puzzles")
    if puzzles.is_empty():
        print()
        continue
    prompt = puzzles.row(0, named=True)["prompt"]
    print(prompt)
    print()


numerals: 1576 puzzles
In Alice's Wonderland, numbers are secretly converted into a different numeral system. Some examples are given below:
11 -> XI
15 -> XV
94 -> XCIV
19 -> XIX
Now, write the number 38 in the Wonderland numeral system.

unit conversion: 1594 puzzles
In Alice's Wonderland, a secret unit conversion is applied to measurements. For example:
10.08 m becomes 6.69
17.83 m becomes 11.83
35.85 m becomes 23.79
17.06 m becomes 11.32
31.54 m becomes 20.93
Now, convert the following measurement: 25.09 m

bit manipulation: 1602 puzzles
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the outp